# 🚁 DIP Project — Gün 1: Ortam Kurulumu
**Runtime → Change runtime type → GPU (T4)**

> **Strateji:** cv2 4.10'a yükselt → numpy 2.x ile uyumlu. Restart gerekmez.
>
> 1. **HÜCRE 1'i** çalıştır (kurulum tamamlanır)
> 2. **HÜCRE 2'den** devam et

## HÜCRE 1 — Kurulum + Auto Restart (bir kez çalıştır)

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(args))

# cv2 4.10 = numpy 2.x resmi desteği var (restart gerekmez)
print('[0/4] cv2 4.10.0.84 kuruluyor (numpy 2.x uyumlu)...')
pip('--force-reinstall', 'opencv-python-headless==4.10.0.84')

print('[1/4] ultralytics, sahi kuruluyor...')
pip('ultralytics==8.2.0', 'sahi==0.11.15', 'gdown', 'PyYAML', 'tqdm')

print('[2/4] albumentations kuruluyor...')
pip('albumentations>=1.3.1,<2.0')

print('[3/4] lapx kuruluyor...')
pip('lapx')
try:
    pip('--no-binary', 'cython_bbox', 'cython_bbox')
    print('✅ cython_bbox kuruldu')
except Exception:
    print('⚠️ cython_bbox atlandı (opsiyonel)')

print()
print('=' * 50)
print('✅ Kurulum tamam! HÜCRE 2den devam et.')
print('=' * 50)

## HÜCRE 2 — GPU Doğrula *(Restart sonrası buradan başla)*

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else '⚠️ GPU yok!')
print(f'Python: {sys.version}')

## HÜCRE 3 — numpy / cv2 Doğrula

In [ ]:
import numpy as np
import cv2
print(f'numpy : {np.__version__}')
print(f'cv2   : {cv2.__version__}')
assert cv2.__version__ >= '4.10', f'❌ Eski cv2 ({cv2.__version__}) — HÜCRE 1i tekrar çalıştır!'
print('✅ cv2 4.10+ | numpy 2.x uyumlu!')

## HÜCRE 4 — ByteTrack Kurulumu

In [ ]:
import os, subprocess, sys
if not os.path.exists('/content/ByteTrack'):
    !git clone -q https://github.com/ifzhang/ByteTrack.git /content/ByteTrack

%cd /content/ByteTrack
# onnxruntime==1.8.0 mevcut değil — requirements'tan çıkar
!grep -v 'onnxruntime' requirements.txt > req_fixed.txt
!pip install -q -r req_fixed.txt
!pip install -q -e .
%cd /content
print('✅ ByteTrack hazır!')

## HÜCRE 5 — PyTorch / CUDA

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## HÜCRE 6 — Versiyon Tablosu

In [ ]:
import importlib
pkgs = ['ultralytics','sahi','torch','torchvision',
        'cv2','numpy','pandas','matplotlib','albumentations','yaml']
print(f'{"Paket":<22} {"Versiyon"}')
print('-'*40)
for p in pkgs:
    try:
        m = importlib.import_module(p)
        print(f'{p:<22} ✅  {getattr(m,"__version__","N/A")}')
    except Exception as e:
        print(f'{p:<22} ❌  {e}')
try:
    import lapx; print(f'{"lapx":<22} ✅')
except ImportError as e:
    print(f'{"lapx":<22} ❌  {e}')

## HÜCRE 7 — Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/DIP_Project'
DATASET_DIR = f'{PROJECT_DIR}/datasets/VisDrone'
os.makedirs(f'{PROJECT_DIR}/datasets', exist_ok=True)
print(f'✅ {PROJECT_DIR}')

## HÜCRE 8 — VisDrone Dataset

In [ ]:
import os
DATASET_DIR = '/content/drive/MyDrive/DIP_Project/datasets/VisDrone'
if os.path.exists(DATASET_DIR) and os.listdir(DATASET_DIR):
    print('✅ VisDrone zaten mevcut.')
else:
    from ultralytics.utils.downloads import download
    os.makedirs(DATASET_DIR, exist_ok=True)
    for url in [
        'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip',
        'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip',
        'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip',
    ]:
        download(url, dir=DATASET_DIR, unzip=True)
    print('✅ VisDrone indirildi!')

## HÜCRE 9 — YOLOv8 Sanity Check

In [ ]:
import torch
from ultralytics import YOLO
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLO('yolov8n.pt')
r = model.predict('https://ultralytics.com/images/bus.jpg',
                  conf=0.25, device=DEVICE, verbose=False)[0]
print(f'📊 {len(r.boxes)} nesne | {r.speed["inference"]:.1f}ms | {DEVICE}')
plt.figure(figsize=(10,6))
plt.imshow(r.plot()[:,:,::-1]); plt.axis('off')
plt.title('YOLOv8n ✅'); plt.show()
print('\n✅ GÜN 1 TAMAMLANDI!')

---
## ✅ Gün 1 Özet
| Paket | Versiyon | Not |
|-------|----------|-----|
| numpy | 2.x (değişmedi) | ✅ |
| opencv-python-headless | **4.9+** | ✅ numpy 2.x uyumlu |
| ultralytics | 8.2.0 | ✅ |
| sahi | 0.11.15 | ✅ |
| albumentations | 2.x | ✅ |
| lapx | latest | ✅ numpy 2.x uyumlu |
| ByteTrack | latest | ✅ |